# Regression model comparison for electrical generation forecasting

This notebook trains and compares several *machine learning* models to forecast
**daily electrical generation** in Colombia (in GWh), using temporal features
derived from the historical series itself.

It complements the Prophet model from `modelos.ipynb` with a supervised regression
approach (lags + calendar) and an honest metric comparison on a temporal test set.

> **Data:** `../data/raw/Generacion.csv` (download from [SIMEM](https://www.simem.co/); see `../data/README.md`).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

RANDOM_STATE = 42

## 1. Loading and daily aggregation

`Generacion.csv` contains multiple records per day (one per plant). We sum the
actual estimated generation by date and convert from kWh to **GWh** (÷ 1e6).

In [ ]:
df = pd.read_csv(
    "../data/raw/Generacion.csv",
    encoding="utf-8-sig",
    usecols=["Fecha", "GeneracionRealEstimada"],
)
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")

# Daily total in GWh, continuous daily frequency and gap interpolation
series = (df.groupby("Fecha")["GeneracionRealEstimada"].sum() / 1e6).rename("Valor")
series = series.asfreq("D").interpolate()

print(f"Period: {series.index.min().date()} -> {series.index.max().date()} | n={len(series)} days")
print(f"Mean generation: {series.mean():.1f} GWh/day")
series.tail()

## 2. Feature engineering

We build predictors from the series: **calendar** (day of week, month, day of
year, weekend), **lags** (1, 7, 14, and 30 days), and **rolling means** (7 and 30 days).
All lags use only past information to avoid data leakage.

In [ ]:
d = pd.DataFrame({"Valor": series})
d["dow"] = d.index.dayofweek
d["month"] = d.index.month
d["doy"] = d.index.dayofyear
d["is_weekend"] = (d["dow"] >= 5).astype(int)

for lag in (1, 7, 14, 30):
    d[f"lag{lag}"] = d["Valor"].shift(lag)
d["roll7"] = d["Valor"].shift(1).rolling(7).mean()
d["roll30"] = d["Valor"].shift(1).rolling(30).mean()

d = d.dropna()
features = [c for c in d.columns if c != "Valor"]
X, y = d[features], d["Valor"]
print(f"Usable observations: {len(d)} | n_features: {len(features)}")
features

## 3. Temporal split and scaling

Since this is a time series, data is **not shuffled**: we train on the first 80 % of
the history and evaluate on the last 20 %. Scaling is fit only on the training set.

In [4]:
cut = int(len(d) * 0.8)
X_train, X_test = X.iloc[:cut], X.iloc[cut:]
y_train, y_test = y.iloc[:cut], y.iloc[cut:]

scaler = StandardScaler().fit(X_train)
X_train_s = pd.DataFrame(scaler.transform(X_train), index=X_train.index, columns=features)
X_test_s = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=features)

print(f"Train: {X_train.index.min().date()} -> {X_train.index.max().date()} ({len(X_train)})")
print(f"Test:  {X_test.index.min().date()} -> {X_test.index.max().date()} ({len(X_test)})")

Train: 2013-01-31 -> 2023-03-30 (3711)
Test:  2023-03-31 -> 2025-10-13 (928)


## 4. Training and model comparison

We compare four models: **Ridge** (regularized linear), **HistGradientBoosting**,
**XGBoost**, and **LightGBM**, with the same features.

In [ ]:
models = {
    "Ridge": Ridge(alpha=1.0),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
    "XGBoost": XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                            subsample=0.9, colsample_bytree=0.9, random_state=RANDOM_STATE),
    "LightGBM": LGBMRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                              random_state=RANDOM_STATE, verbose=-1),
}

rows, predictions = [], {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    pred = model.predict(X_test_s)
    predictions[name] = pred
    rows.append({
        "Modelo": name,
        "R2": r2_score(y_test, pred),
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": root_mean_squared_error(y_test, pred),
    })

metrics = pd.DataFrame(rows).sort_values("R2", ascending=False).reset_index(drop=True)
metrics.round(3)

## 5. Visualization: actual vs. predicted (best model)

In [ ]:
best_name = metrics.iloc[0]["Modelo"]
best_model = models[best_name]

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(y_test.index, y_test.values, label="Actual", color="black", linewidth=1.2)
ax.plot(y_test.index, predictions[best_name], label=f"Predicted ({best_name})",
        color="tab:red", linewidth=1.2, alpha=0.8)
ax.set_title(f"Daily generation: actual vs. predicted — {best_name} "
             f"(R2={metrics.iloc[0]['R2']:.3f})")
ax.set_xlabel("Date"); ax.set_ylabel("Generation (GWh)")
ax.legend(); fig.tight_layout()
fig.savefig("../results/figures/generacion_ml_real_vs_pred.png", dpi=120)
plt.show()

## 6. Saving the best model, scaler, and metrics

**Reproducible** artifacts are persisted from this notebook (unlike external models
without code that generates them).

In [ ]:
joblib.dump(best_model, "../models/generacion_mejor_modelo.joblib")
joblib.dump(scaler, "../models/generacion_scaler.joblib")
metrics.to_csv("../results/metricas_generacion_ml.csv", index=False)
print(f"Best model: {best_name}")
print("Saved: ../models/generacion_mejor_modelo.joblib, ../models/generacion_scaler.joblib")
print("Metrics: ../results/metricas_generacion_ml.csv")

## 7. Conclusions

- **Ridge with lags** captures well the strong autocorrelation of daily generation,
  achieving the best performance on the test set.
- The *boosting* models (XGBoost, LightGBM, HistGradientBoosting) do not outperform
  the linear model here: with a highly autoregressive series and few exogenous variables,
  linear regularization is more robust and less prone to overfitting.
- **Next steps:** incorporate exogenous variables (climate, holidays, economic activity),
  temporal cross-validation with `TimeSeriesSplit`, and hyperparameter tuning.